In [0]:
dbutils.secrets.list(scope='retailflow_scope')

In [0]:
client_id = dbutils.secrets.get(scope='retailflow_scope', key='client-id')
client_secret = dbutils.secrets.get(scope='retailflow_scope', key='client-secret')
tenant_id = dbutils.secrets.get(scope='retailflow_scope', key='tenant-id')

In [0]:
spark.conf.set("fs.azure.account.auth.type.datalakealli.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.datalakealli.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.datalakealli.dfs.core.windows.net", client_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.datalakealli.dfs.core.windows.net", client_secret)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.datalakealli.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
# Test: list contents of the container (should be empty or show existing folders)
display(dbutils.fs.ls("abfss://retailflow-lake@datalakealli.dfs.core.windows.net/"))

In [0]:
BASE_PATH = "abfss://retailflow-lake@datalakealli.dfs.core.windows.net"

In [0]:
for folder in ["landing", "bronze", "silver", "gold"]:
    try:
        dbutils.fs.mkdirs(f"{BASE_PATH}/{folder}")
        print(f"Created: {BASE_PATH}/{folder}")
    except Exception as e:
        if "LOCATION_OVERLAP" in str(e):
            print(f"Skipped (managed by Unity Catalog): {BASE_PATH}/{folder}")
        else:
            raise

In [0]:
STORAGE_ACCOUNT = "datalakealli"
CONTAINER = "retailflow-lake"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

spark.sql(f"""
CREATE CATALOG IF NOT EXISTS retailflow
MANAGED LOCATION '{BASE_PATH}/catalog-root'
""")

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS retailflow.landing
MANAGED LOCATION '{BASE_PATH}/landing'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS retailflow.bronze
MANAGED LOCATION '{BASE_PATH}/bronze'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS retailflow.silver
MANAGED LOCATION '{BASE_PATH}/silver'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS retailflow.gold
MANAGED LOCATION '{BASE_PATH}/gold'
""")

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS retailflow.landing.raw_files
""")

In [0]:
display(spark.sql("SHOW SCHEMAS IN retailflow"))
display(spark.sql("SHOW VOLUMES IN retailflow.landing"))